# Chương 1 bài 1 Thống kê mô tả rượu đỏ

**Nguồn:** Lab EDA, mục 1.1.3, trang 3

**Đề bài:** Thực hiện thống kê mô tả trên tập dữ liệu phân loại chất lượng rượu đỏ.

Code triển khai bài tập theo dữ liệu đã cung cấp; không phải đoạn chép nguyên văn bài mẫu.

In [1]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data').is_dir() and (p / 'README.md').is_file())
DATA = ROOT / 'data'
OUT = ROOT / 'ket_qua' / 'C1_BT01_ThongKe_RuouDo'
OUT.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (8, 4)})

def save_fig(name):
    plt.tight_layout()
    plt.savefig(OUT / name, dpi=140, bbox_inches='tight')
    plt.show()

def save_summary(values):
    (OUT / 'tom_tat.json').write_text(
        json.dumps(values, ensure_ascii=False, indent=2), encoding='utf-8'
    )


## 1 Nạp và kiểm tra dữ liệu

In [2]:
df = pd.read_csv(DATA / 'winequality-red.csv', sep=None, engine='python')
print(df.shape)
print(df.head())
print('Duplicates:', df.duplicated().sum())
print('Missing:', df.isna().sum().sum())

(1599, 12)
   fixed acidity  volatile acidity  citric acid  ...  sulphates  alcohol  quality
0            7.4              0.70         0.00  ...       0.56      9.4        5
1            7.8              0.88         0.00  ...       0.68      9.8        5
2            7.8              0.76         0.04  ...       0.65      9.8        5
3           11.2              0.28         0.56  ...       0.58      9.8        6
4            7.4              0.70         0.00  ...       0.56      9.4        5

[5 rows x 12 columns]
Duplicates: 240
Missing: 0


## 2 Tính các thước đo thống kê

In [3]:
numeric = df.select_dtypes(include='number')
summary = numeric.describe().T
summary['mode_first'] = numeric.mode().iloc[0]
summary['variance_sample'] = numeric.var(ddof=1)
summary['std_sample'] = numeric.std(ddof=1)
summary['variance_population'] = numeric.var(ddof=0)
summary['std_population'] = numeric.std(ddof=0)
summary['range'] = numeric.max() - numeric.min()
summary['percentile_60'] = numeric.quantile(0.60)
summary['IQR'] = numeric.quantile(0.75) - numeric.quantile(0.25)
summary['missing'] = numeric.isna().sum()
q1, q3 = numeric.quantile(0.25), numeric.quantile(0.75)
iqr = q3 - q1
summary['outside_IQR_fences'] = ((numeric < q1 - 1.5 * iqr) |
                                 (numeric > q3 + 1.5 * iqr)).sum()
print(summary.round(4).to_string())
summary.to_csv(OUT / 'thong_ke_mo_ta.csv', encoding='utf-8-sig')

                       count     mean      std     min      25%      50%      75%       max  mode_first  variance_sample  std_sample  variance_population  std_population     range  percentile_60      IQR  missing  outside_IQR_fences
fixed acidity         1599.0   8.3196   1.7411  4.6000   7.1000   7.9000   9.2000   15.9000      7.2000           3.0314      1.7411               3.0295          1.7406   11.3000         8.3000   2.1000        0                  49
volatile acidity      1599.0   0.5278   0.1791  0.1200   0.3900   0.5200   0.6400    1.5800      0.6000           0.0321      0.1791               0.0320          0.1790    1.4600         0.5700   0.2500        0                  19
citric acid           1599.0   0.2710   0.1948  0.0000   0.0900   0.2600   0.4200    1.0000      0.0000           0.0379      0.1948               0.0379          0.1947    1.0000         0.3200   0.3300        0                   1
residual sugar        1599.0   2.5388   1.4099  0.9000   1.9000   2.

## 3 Phân bố nhãn và kết quả chính

In [4]:
counts = df['quality'].value_counts().sort_index()
print('Quality counts:\n', counts)
counts.to_csv(OUT / 'phan_bo_quality.csv')
save_summary({'rows': len(df), 'columns': df.shape[1],
              'duplicate_rows': int(df.duplicated().sum()),
              'missing_cells': int(df.isna().sum().sum()),
              'quality_counts': {str(k):int(v) for k,v in counts.items()},
              'alcohol_mean': float(df['alcohol'].mean()),
              'alcohol_median': float(df['alcohol'].median())})

Quality counts:
 quality
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64


## Nhận xét và giới hạn
Bảng có phương sai/độ lệch chuẩn mẫu (ddof=1) và tổng thể (ddof=0). Quality là nhãn có thứ bậc, cần xem tần số bên cạnh trung bình. Dòng trùng được đếm, không mặc nhiên xóa vì chưa có ID chứng minh cùng một quan sát. Quy tắc IQR chỉ đánh dấu ứng viên ngoại lai, không tự loại dữ liệu.